In [13]:
!pip install /kaggle/input/rdkit-2025-3-3-cp311/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl
!pip install mordred --no-index --find-links=file:///kaggle/input/mordred-1-2-0-py3-none-any/
!pip install torch_geometric --no-index --find-links=file:/kaggle/input/torch-geometric-2-6-1-whl/torch_geometric-2.6.1-py3-none-any.whl

Processing /kaggle/input/rdkit-2025-3-3-cp311/rdkit-2025.3.3-cp311-cp311-manylinux_2_28_x86_64.whl
rdkit is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
Looking in links: file:///kaggle/input/mordred-1-2-0-py3-none-any/
Looking in links: file:///kaggle/input/torch-geometric-2-6-1-whl/torch_geometric-2.6.1-py3-none-any.whl


In [14]:
import warnings
warnings.simplefilter(action='ignore')

import os
import re
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool, global_max_pool, global_add_pool
from torch_geometric.nn import BatchNorm, LayerNorm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib

# Load data (same as original code)
train = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/train.csv')
test = pd.read_csv('/kaggle/input/neurips-open-polymer-prediction-2025/test.csv')

def improved_tokenize_smiles(smiles):
    """Same tokenization as original code"""
    pattern = r'(\[[^\[\]]{1,10}\]|Br|Cl|[bcnops]|[BCNOFPSI]|[()=+\-#@:/\\]|\d+|\.)'
    tokens = re.findall(pattern, smiles)
    reconstructed = ''.join(tokens)
    remaining = smiles.replace(reconstructed, '')
    if remaining:
        tokens.extend(list(remaining))
    return tokens

class SMILESToGraph:
    """Convert SMILES to molecular graph representation"""
    
    def __init__(self):
        # Atom feature mapping
        self.atom_vocab = {
            'C': 0, 'N': 1, 'O': 2, 'S': 3, 'P': 4, 'F': 5, 'Cl': 6, 'Br': 7, 'I': 8,
            'c': 9, 'n': 10, 'o': 11, 's': 12, 'p': 13, 'H': 14, '[': 15, ']': 16,
            'UNK': 17
        }
        
        # Bond type mapping
        self.bond_vocab = {
            'SINGLE': 0, 'DOUBLE': 1, 'TRIPLE': 2, 'AROMATIC': 3, 'UNK': 4
        }
        
        # Electronegativity values for node features
        self.electronegativity = {
            'C': 2.55, 'N': 3.04, 'O': 3.44, 'S': 2.58, 'P': 2.19, 
            'F': 3.98, 'Cl': 3.16, 'Br': 2.96, 'I': 2.66, 'H': 2.20,
            'c': 2.55, 'n': 3.04, 'o': 3.44, 's': 2.58, 'p': 2.19
        }
        
    def get_atom_features(self, atom_token):
        """Extract features for each atom"""
        features = []
        
        # Atom type (one-hot encoded)
        atom_type = self.atom_vocab.get(atom_token, self.atom_vocab['UNK'])
        atom_onehot = [0] * len(self.atom_vocab)
        atom_onehot[atom_type] = 1
        features.extend(atom_onehot)
        
        # Electronegativity
        electronegativity = self.electronegativity.get(atom_token, 2.5)
        features.append(electronegativity)
        
        # Is aromatic
        is_aromatic = 1 if atom_token.islower() else 0
        features.append(is_aromatic)
        
        # Is heteroatom
        is_heteroatom = 1 if atom_token.upper() in ['N', 'O', 'S', 'P'] else 0
        features.append(is_heteroatom)
        
        return features
    
    def detect_bonds(self, tokens):
        """Detect bonds between atoms in tokenized SMILES"""
        edges = []
        edge_attrs = []
        atom_positions = []
        
        # Find atom positions
        for i, token in enumerate(tokens):
            if token in self.atom_vocab and token not in ['[', ']', '(', ')']:
                atom_positions.append(i)
        
        # Simple bond detection based on adjacency and bond symbols
        for i in range(len(atom_positions) - 1):
            pos1, pos2 = atom_positions[i], atom_positions[i + 1]
            
            # Check for bond symbols between atoms
            bond_type = 'SINGLE'  # default
            
            # Look for bond indicators between positions
            between_tokens = tokens[pos1 + 1:pos2]
            
            if '=' in between_tokens:
                bond_type = 'DOUBLE'
            elif '#' in between_tokens:
                bond_type = 'TRIPLE'
            elif any(t.islower() for t in between_tokens):
                bond_type = 'AROMATIC'
            
            # Add edge in both directions (undirected graph)
            edges.extend([[i, i + 1], [i + 1, i]])
            bond_idx = self.bond_vocab.get(bond_type, self.bond_vocab['UNK'])
            edge_attrs.extend([bond_idx, bond_idx])
        
        # Handle ring connections (simplified)
        ring_numbers = {}
        for i, (token_idx, token) in enumerate(zip(atom_positions, [tokens[pos] for pos in atom_positions])):
            # Look for ring numbers after atoms
            next_tokens = tokens[token_idx + 1:token_idx + 3] if token_idx + 1 < len(tokens) else []
            for next_token in next_tokens:
                if next_token.isdigit():
                    ring_num = next_token
                    if ring_num in ring_numbers:
                        # Connect to previous atom with same ring number
                        prev_atom_idx = ring_numbers[ring_num]
                        edges.extend([[i, prev_atom_idx], [prev_atom_idx, i]])
                        edge_attrs.extend([self.bond_vocab['SINGLE'], self.bond_vocab['SINGLE']])
                        del ring_numbers[ring_num]
                    else:
                        ring_numbers[ring_num] = i
        
        return edges, edge_attrs
    
    def smiles_to_graph(self, smiles):
        """Convert SMILES string to PyTorch Geometric Data object"""
        tokens = improved_tokenize_smiles(smiles)
        
        # Extract atoms (filter out non-atom tokens)
        atoms = []
        for token in tokens:
            if token in self.atom_vocab and token not in ['[', ']', '(', ')']:
                atoms.append(token)
        
        if len(atoms) == 0:
            # Handle edge case: create single dummy node
            atoms = ['C']
        
        # Create node features
        node_features = []
        for atom in atoms:
            features = self.get_atom_features(atom)
            node_features.append(features)
        
        # Detect edges and edge attributes
        edges, edge_attrs = self.detect_bonds(tokens)
        
        # Convert to tensors
        x = torch.tensor(node_features, dtype=torch.float)
        
        if len(edges) > 0:
            edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
            edge_attr = torch.tensor(edge_attrs, dtype=torch.float).unsqueeze(1)
        else:
            # No edges - create self-loops
            num_nodes = len(atoms)
            edge_index = torch.tensor([[i, i] for i in range(num_nodes)], dtype=torch.long).t().contiguous()
            edge_attr = torch.zeros((num_nodes, 1), dtype=torch.float)
        
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

class PolymerGNN(nn.Module):
    """Graph Neural Network for polymer property prediction"""
    
    def __init__(self, input_dim, hidden_dim=128, num_layers=4, num_targets=5, dropout=0.2):
        super(PolymerGNN, self).__init__()
        
        self.num_targets = num_targets
        self.num_layers = num_layers
        
        # Graph convolution layers
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        
        # First layer
        self.convs.append(GATConv(input_dim, hidden_dim, heads=4, concat=False, dropout=dropout))
        self.batch_norms.append(BatchNorm(hidden_dim))
        
        # Hidden layers
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(hidden_dim, hidden_dim, heads=4, concat=False, dropout=dropout))
            self.batch_norms.append(BatchNorm(hidden_dim))
        
        # Last graph conv layer
        self.convs.append(GCNConv(hidden_dim, hidden_dim))
        self.batch_norms.append(BatchNorm(hidden_dim))
        
        # Global pooling and final prediction layers
        self.dropout = nn.Dropout(dropout)
        
        # Multi-head attention for graph-level features
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=4, dropout=dropout, batch_first=True)
        
        # Final prediction heads for each target
        self.predictors = nn.ModuleList()
        for _ in range(num_targets):
            predictor = nn.Sequential(
                nn.Linear(hidden_dim * 3, hidden_dim),  # 3 pooling methods
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim // 2, 1)
            )
            self.predictors.append(predictor)
    
    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        
        # Graph convolutions
        for i in range(self.num_layers):
            if i < self.num_layers - 1:
                # GAT layers
                x = self.convs[i](x, edge_index)
            else:
                # Final GCN layer
                x = self.convs[i](x, edge_index)
            
            x = self.batch_norms[i](x)
            x = F.relu(x)
            x = self.dropout(x)
        
        # Global pooling
        mean_pool = global_mean_pool(x, batch)
        max_pool = global_max_pool(x, batch)
        add_pool = global_add_pool(x, batch)
        
        # Combine pooling methods
        graph_repr = torch.cat([mean_pool, max_pool, add_pool], dim=1)
        
        # Multi-target prediction
        predictions = []
        for predictor in self.predictors:
            pred = predictor(graph_repr)
            predictions.append(pred)
        
        return torch.cat(predictions, dim=1)

def create_data_loaders(smiles_list, targets=None, batch_size=32, shuffle=True):
    """Create PyTorch Geometric data loaders with error handling"""
    converter = SMILESToGraph()
    
    data_list = []
    failed_conversions = 0
    
    for i, smiles in enumerate(smiles_list):
        try:
            graph = converter.smiles_to_graph(smiles)
            
            # Validate graph
            if graph.x.shape[0] == 0:
                print(f"Warning: Empty graph for SMILES {i}: {smiles}")
                continue
                
            if targets is not None:
                # Ensure targets are valid (no NaN)
                target_row = targets[i]
                if np.any(np.isnan(target_row)):
                    print(f"Warning: NaN targets for SMILES {i}: {smiles}")
                    continue
                    
                y = torch.tensor(target_row, dtype=torch.float).view(1, -1)
                graph.y = y
            
            data_list.append(graph)
            
        except Exception as e:
            failed_conversions += 1
            print(f"Failed to convert SMILES {i}: {smiles}, Error: {e}")
            continue
    
    print(f"Successfully converted {len(data_list)} graphs, failed: {failed_conversions}")
    
    if len(data_list) == 0:
        raise ValueError("No valid graphs created from SMILES list")
    
    return DataLoader(data_list, batch_size=batch_size, shuffle=shuffle)

def train_gnn_model(train_smiles, train_targets, val_smiles, val_targets, 
                   num_epochs=200, batch_size=32, learning_rate=0.001):
    """Train the GNN model with robust error handling"""
    
    print("Creating data loaders...")
    # Create data loaders with error handling
    try:
        train_loader = create_data_loaders(train_smiles, train_targets, batch_size, shuffle=True)
        val_loader = create_data_loaders(val_smiles, val_targets, batch_size, shuffle=False)
    except Exception as e:
        print(f"Error creating data loaders: {e}")
        raise
    
    print(f"Created train loader with {len(train_loader.dataset)} samples")
    print(f"Created val loader with {len(val_loader.dataset)} samples")
    
    # Initialize model
    # Get input dimension from first graph
    try:
        sample_graph = SMILESToGraph().smiles_to_graph(train_smiles[0])
        input_dim = sample_graph.x.shape[1]
        print(f"Input dimension: {input_dim}")
    except Exception as e:
        print(f"Error getting input dimension: {e}")
        # Use a reasonable default based on our feature extraction
        input_dim = 33  # Based on atom vocab size + additional features
        print(f"Using default input dimension: {input_dim}")
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    model = PolymerGNN(input_dim=input_dim, hidden_dim=128, num_layers=4, 
                      num_targets=5, dropout=0.2).to(device)
    
    # Loss and optimizer
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', 
                                                          factor=0.7, patience=10, verbose=True)
    
    # Training loop
    best_val_loss = float('inf')
    best_model_state = None
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_losses = []
        
        for batch_idx, batch in enumerate(train_loader):
            try:
                batch = batch.to(device)
                optimizer.zero_grad()
                
                # Forward pass
                predictions = model(batch)
                loss = criterion(predictions, batch.y)
                
                # Backward pass
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                
                train_losses.append(loss.item())
                
            except Exception as e:
                print(f"Error in training batch {batch_idx}: {e}")
                continue
        
        if len(train_losses) == 0:
            print("No successful training batches!")
            break
        
        # Validation
        model.eval()
        val_losses = []
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            for batch_idx, batch in enumerate(val_loader):
                try:
                    batch = batch.to(device)
                    predictions = model(batch)
                    loss = criterion(predictions, batch.y)
                    val_losses.append(loss.item())
                    
                    all_preds.append(predictions.cpu().numpy())
                    all_targets.append(batch.y.cpu().numpy())
                    
                except Exception as e:
                    print(f"Error in validation batch {batch_idx}: {e}")
                    continue
        
        if len(val_losses) == 0:
            print("No successful validation batches!")
            continue
        
        # Calculate metrics
        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)
        
        scheduler.step(val_loss)
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
        
        if epoch % 20 == 0:
            print(f'Epoch {epoch:03d}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    else:
        print("Warning: No best model state saved!")
    
    return model, best_val_loss

def predict_with_gnn(model, test_smiles, batch_size=32):
    """Make predictions using trained GNN"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    
    test_loader = create_data_loaders(test_smiles, targets=None, batch_size=batch_size, shuffle=False)
    
    all_predictions = []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            predictions = model(batch)
            all_predictions.append(predictions.cpu().numpy())
    
    return np.vstack(all_predictions)

# ========== MAIN TRAINING PIPELINE ==========

print("Preparing data for GNN training...")

# Prepare training data
train_clean = train.dropna(subset=['Tg'])
train_smiles = train_clean['SMILES'].tolist()
train_targets = train_clean[['Tg']].values

# Normalize targets
target_scaler = StandardScaler()
train_targets_scaled = target_scaler.fit_transform(train_targets)

# Train-validation split
train_smiles_split, val_smiles_split, train_targets_split, val_targets_split = train_test_split(
    train_smiles, train_targets_scaled, test_size=0.1, random_state=42
)

print(f"Training samples: {len(train_smiles_split)}")
print(f"Validation samples: {len(val_smiles_split)}")
print(f"Test samples: {len(test)}")

# Train GNN model
print("Training GNN model...")
gnn_model, best_val_loss = train_gnn_model(
    train_smiles_split, train_targets_split,
    val_smiles_split, val_targets_split,
    num_epochs=150, batch_size=32, learning_rate=0.001
)

print(f"Best validation loss: {best_val_loss:.4f}")

# Make predictions on test set
print("Making predictions on test set...")
test_smiles = test['SMILES'].tolist()
test_predictions_scaled = predict_with_gnn(gnn_model, test_smiles)

# Inverse transform predictions
test_predictions = target_scaler.inverse_transform(test_predictions_scaled)

# Create submission
submission = pd.DataFrame({
    'id': test['id'],
    'Tg': test_predictions[:, 0],
    'FFV': test_predictions[:, 1],
    'Tc': test_predictions[:, 2],
    'Density': test_predictions[:, 3],
    'Rg': test_predictions[:, 4]
})

# Save submission
submission.to_csv("gnn_submission.csv", index=False)
print("GNN submission file saved as 'gnn_submission.csv'")

# Save model and scaler
torch.save(gnn_model.state_dict(), 'polymer_gnn_model.pth')
joblib.dump(target_scaler, 'target_scaler.pkl')
print("Model and scaler saved!")

# Print some validation metrics
print("\nValidation predictions vs actual (first 10 samples):")
val_predictions_scaled = predict_with_gnn(gnn_model, val_smiles_split[:10])
val_predictions = target_scaler.inverse_transform(val_predictions_scaled)
val_actual = target_scaler.inverse_transform(val_targets_split[:10])

target_names = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']
for i, name in enumerate(target_names):
    mse = mean_squared_error(val_actual[:, i], val_predictions[:, i])
    mae = mean_absolute_error(val_actual[:, i], val_predictions[:, i])
    print(f"{name} - MSE: {mse:.4f}, MAE: {mae:.4f}")

Preparing data for GNN training...
Training samples: 459
Validation samples: 52
Test samples: 3
Training GNN model...
Creating data loaders...
Successfully converted 459 graphs, failed: 0
Successfully converted 52 graphs, failed: 0
Created train loader with 459 samples
Created val loader with 52 samples
Input dimension: 21
Using device: cpu
Epoch 000, Train Loss: 1.7181, Val Loss: 1.0616
Epoch 020, Train Loss: 0.4741, Val Loss: 0.4734
Epoch 040, Train Loss: 0.4229, Val Loss: 0.4059
Epoch 060, Train Loss: 0.3663, Val Loss: 0.3940
Epoch 080, Train Loss: 0.3626, Val Loss: 0.4481
Epoch 100, Train Loss: 0.3063, Val Loss: 0.4679
Epoch 120, Train Loss: 0.2875, Val Loss: 0.4432
Epoch 140, Train Loss: 0.2903, Val Loss: 0.4489
Best validation loss: 0.3435
Making predictions on test set...
Successfully converted 3 graphs, failed: 0
GNN submission file saved as 'gnn_submission.csv'
Model and scaler saved!

Validation predictions vs actual (first 10 samples):
Successfully converted 10 graphs, faile

IndexError: index 1 is out of bounds for axis 1 with size 1

In [15]:
test_predictions

array([[128.5247  , 140.6262  , 134.98746 , 122.396416, 140.14957 ],
       [160.66922 , 172.28375 , 173.55214 , 166.22641 , 152.6521  ],
       [ 92.781944,  96.30674 ,  92.284325,  91.13422 , 107.126   ]],
      dtype=float32)